# Safe-SRE OpenEnv — Reproducible Training (Colab)

Reviewer-runnable notebook that clones the env from GitHub, installs the stack, and runs a short GRPO training run on a Colab T4 / L4 GPU.

**Runtime:** Colab T4 GPU (free tier works) or L4 (Pro). Total time ~25 min for `--max_steps 50` (a smoke), ~3-4 hours for a full `--max_steps 400` run.

**Model:** `Qwen/Qwen3-1.7B`, 4-bit base + LoRA via Unsloth, colocated vLLM.

**Scenarios:** 25 incidents across 6 SRE failure categories (service / disk / process / permissions / network / db_recovery). Held-out: 6 adversarial + 2 compound for eval.

**Reward:** 5 composable functions (safety / correctness / minimality / format / investigation) — see `core/rewards.py`.

## 1. Clone the repo + install

In [ ]:
# 30 sec: clone the source.
!git clone https://github.com/dhruv608/devops-model /content/safe_sre_env
%cd /content/safe_sre_env

In [ ]:
# ~5-10 min: install heavy deps. unsloth must come BEFORE trl/transformers.
!pip install -q uv
!uv pip install --system -e . trl datasets wandb bashlex pytest
!uv pip install --system unsloth
!uv pip install --system vllm

In [ ]:
# Sanity check: 88 unit + contract tests cover the env, bash classifier,
# rewards, scenarios, and the train script's --dry_run config.
!PYTHONPATH=. python -m pytest tests/ -q

## 2. Verify the env without training (mock rollouts)

These commands run end-to-end with **no model weights loaded** — just deterministic stand-in generators (`MockGenerator(style='impulsive')` vs `MockGenerator(style='cautious')`) so we can prove the env's reward signal is wired correctly.

Expected headline on the held-out eval set:
- `safety_violation_rate: 4.2% → 0%`
- `mean_total_reward: +4.61 → +5.00`
- on `adv_var_log_full_with_live_app`: impulsive `rm -rf /var/log/*` gets **−3.65**, cautious scoped fix gets **+5.75** (delta **+9.4**).


In [ ]:
!PYTHONPATH=. python -m eval.eval --mock --episodes-per-scenario 3 --out plots/eval_mock.json

In [ ]:
!PYTHONPATH=. python -m demo.replay --mock --out demo/before_after.md
from IPython.display import Markdown
Markdown(open('demo/before_after.md').read())

## 3. Inspect the GRPO config without training

Prints the model id, full TRL `GRPOConfig`, dataset preview, and reward-function list. Useful for sanity-checking before kicking off a real run.

In [ ]:
!PYTHONPATH=. python train/train_grpo.py --max_steps 50 --dry_run

## 4. Authenticate with HF Hub (so we can pull Qwen3-1.7B + push the trained checkpoint)

Get a write-scope token at https://huggingface.co/settings/tokens. Make sure you've claimed your hackathon $30 credit at https://huggingface.co/coupons/claim/hf-openenv-community first.

In [ ]:
from huggingface_hub import login
import getpass
login(getpass.getpass('HF write token: '), add_to_git_credential=True)

## 5. Short smoke training run (50 steps, ~20 min on T4)

Confirms the GRPO loop produces non-NaN losses and all 5 reward components log correctly. **Drop the `--push_to_hub` flag** if you don't want the smoke checkpoint on the Hub.

In [ ]:
!PYTHONPATH=. python -u train/train_grpo.py \
    --max_steps 50 \
    --num_generations 2 \
    --max_completion_length 1024 \
    --vllm_gpu_memory_utilization 0.10 \
    --report_to none

## 6. Full training run (400 steps, ~3-4 hours)

Same script, more steps, pushed to the Hub. Default hyperparameters mirror `strategy.md §5.1`.

In [ ]:
!PYTHONPATH=. python -u train/train_grpo.py \
    --max_steps 400 \
    --push_to_hub \
    --hub_model_id YOUR_USERNAME/safe-sre-grpo-Qwen3-1.7B \
    --report_to wandb

## 7. Eval the trained checkpoint vs the base

Once training completes, replace `--trained-model` below with whatever Hub repo you pushed to. Produces `plots/eval_results.json` and prints the headline numbers.

In [ ]:
!PYTHONPATH=. python -m eval.eval \
    --base-model Qwen/Qwen3-1.7B \
    --trained-model YOUR_USERNAME/safe-sre-grpo-Qwen3-1.7B \
    --episodes-per-scenario 5 \
    --out plots/eval_results.json